In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

conn = sqlite3.connect("../data/raw/database.sqlite")
raw_path = Path("../data/raw")
cleaned_path = Path("../data/cleaned")
cleaned_path.mkdir(parents=True, exist_ok=True)

In [2]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

In [3]:
country = pd.read_sql_query("""
SELECT id AS country_id, name AS country_name
FROM country
""", conn)

league = pd.read_sql_query("""
SELECT id AS league_id, country_id, name AS league_name
FROM league
""", conn)

team = pd.read_sql_query("""
SELECT team_api_id, team_fifa_api_id, team_long_name, team_short_name
FROM team
""", conn)

match = pd.read_sql_query("""
SELECT
    id AS match_id,
    country_id,
    league_id,
    season,
    stage,
    date,
    match_api_id,
    home_team_api_id,
    away_team_api_id,
    home_team_goal,
    away_team_goal
FROM "match"
""", conn)

In [4]:
country = clean_columns(country)
league = clean_columns(league)
team = clean_columns(team)
match = clean_columns(match)

In [5]:
datasets = {
    "country": country,
    "league": league,
    "team": team,
    "match": match
}

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head())


--- COUNTRY ---
Shape: (11, 2)
Columns: ['country_id', 'country_name']
   country_id country_name
0           1      Belgium
1        1729      England
2        4769       France
3        7809      Germany
4       10257        Italy

--- LEAGUE ---
Shape: (11, 3)
Columns: ['league_id', 'country_id', 'league_name']
   league_id  country_id             league_name
0          1           1  Belgium Jupiler League
1       1729        1729  England Premier League
2       4769        4769          France Ligue 1
3       7809        7809   Germany 1. Bundesliga
4      10257       10257           Italy Serie A

--- TEAM ---
Shape: (299, 4)
Columns: ['team_api_id', 'team_fifa_api_id', 'team_long_name', 'team_short_name']
   team_api_id  team_fifa_api_id     team_long_name team_short_name
0         9987             673.0           KRC Genk             GEN
1         9993             675.0       Beerschot AC             BAC
2        10000           15005.0   SV Zulte-Waregem             ZUL
3    

In [6]:
match["date"] = pd.to_datetime(match["date"], errors="coerce")
match["home_team_goal"] = pd.to_numeric(match["home_team_goal"], errors="coerce")
match["away_team_goal"] = pd.to_numeric(match["away_team_goal"], errors="coerce")

In [7]:
for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    print(df.isnull().sum())


Missing values in country:
country_id      0
country_name    0
dtype: int64

Missing values in league:
league_id      0
country_id     0
league_name    0
dtype: int64

Missing values in team:
team_api_id          0
team_fifa_api_id    11
team_long_name       0
team_short_name      0
dtype: int64

Missing values in match:
match_id            0
country_id          0
league_id           0
season              0
stage               0
date                0
match_api_id        0
home_team_api_id    0
away_team_api_id    0
home_team_goal      0
away_team_goal      0
dtype: int64


In [8]:
league_clean = league.merge(country, on="country_id", how="left")
league_clean.head()

,league_id,country_id,league_name,country_name
0,1,1,Belgium Jupiler League,Belgium
1,1729,1729,England Premier League,England
2,4769,4769,France Ligue 1,France
3,7809,7809,Germany 1. Bundesliga,Germany
4,10257,10257,Italy Serie A,Italy


In [9]:
df = match.merge(
    league_clean[["league_id", "league_name", "country_id", "country_name"]],
    on=["league_id", "country_id"],
    how="left"
)

df.head()

,match_id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,league_name,country_name
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,1,Belgium Jupiler League,Belgium
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,0,Belgium Jupiler League,Belgium
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,3,Belgium Jupiler League,Belgium
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,0,Belgium Jupiler League,Belgium
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,3,Belgium Jupiler League,Belgium


In [10]:
home_team = team[["team_api_id", "team_long_name", "team_short_name"]].copy()
home_team = home_team.rename(columns={
    "team_api_id": "home_team_api_id",
    "team_long_name": "home_team_name",
    "team_short_name": "home_team_short_name"
})

df = df.merge(home_team, on="home_team_api_id", how="left")
df.head()

,match_id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,league_name,country_name,home_team_name,home_team_short_name
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,1,Belgium Jupiler League,Belgium,KRC Genk,GEN
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,0,Belgium Jupiler League,Belgium,SV Zulte-Waregem,ZUL
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,3,Belgium Jupiler League,Belgium,KSV Cercle Brugge,CEB
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,0,Belgium Jupiler League,Belgium,KAA Gent,GEN
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,3,Belgium Jupiler League,Belgium,FCV Dender EH,DEN


In [11]:
away_team = team[["team_api_id", "team_long_name", "team_short_name"]].copy()
away_team = away_team.rename(columns={
    "team_api_id": "away_team_api_id",
    "team_long_name": "away_team_name",
    "team_short_name": "away_team_short_name"
})

df = df.merge(away_team, on="away_team_api_id", how="left")
df.head()

,match_id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,league_name,country_name,home_team_name,home_team_short_name,away_team_name,away_team_short_name
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,1,Belgium Jupiler League,Belgium,KRC Genk,GEN,Beerschot AC,BAC
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,0,Belgium Jupiler League,Belgium,SV Zulte-Waregem,ZUL,Sporting Lokeren,LOK
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,3,Belgium Jupiler League,Belgium,KSV Cercle Brugge,CEB,RSC Anderlecht,AND
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,0,Belgium Jupiler League,Belgium,KAA Gent,GEN,RAEC Mons,MON
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,3,Belgium Jupiler League,Belgium,FCV Dender EH,DEN,Standard de Liège,STL


In [12]:
df["result"] = np.where(
    df["home_team_goal"] > df["away_team_goal"], "Home Win",
    np.where(df["home_team_goal"] < df["away_team_goal"], "Away Win", "Draw")
)

df["goal_diff"] = df["home_team_goal"] - df["away_team_goal"]
df["total_goals"] = df["home_team_goal"] + df["away_team_goal"]

df["home_points"] = np.where(
    df["result"] == "Home Win", 3,
    np.where(df["result"] == "Draw", 1, 0)
)

df["away_points"] = np.where(
    df["result"] == "Away Win", 3,
    np.where(df["result"] == "Draw", 1, 0)
)

In [13]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nResult counts:")
print(df["result"].value_counts())

df.head()

Final shape: (25979, 22)

Missing values:
match_id                0
country_id              0
league_id               0
season                  0
stage                   0
date                    0
match_api_id            0
home_team_api_id        0
away_team_api_id        0
home_team_goal          0
away_team_goal          0
league_name             0
country_name            0
home_team_name          0
home_team_short_name    0
away_team_name          0
away_team_short_name    0
result                  0
goal_diff               0
total_goals             0
home_points             0
away_points             0
dtype: int64

Result counts:
result
Home Win    11917
Away Win     7466
Draw         6596
Name: count, dtype: int64


,match_id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,country_name,home_team_name,home_team_short_name,away_team_name,away_team_short_name,result,goal_diff,total_goals,home_points,away_points
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,...,Belgium,KRC Genk,GEN,Beerschot AC,BAC,Draw,0,2,1,1
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,...,Belgium,SV Zulte-Waregem,ZUL,Sporting Lokeren,LOK,Draw,0,0,1,1
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,...,Belgium,KSV Cercle Brugge,CEB,RSC Anderlecht,AND,Away Win,-3,3,0,3
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,...,Belgium,KAA Gent,GEN,RAEC Mons,MON,Home Win,5,5,3,0
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,...,Belgium,FCV Dender EH,DEN,Standard de Liège,STL,Away Win,-2,4,0,3


In [14]:
df.to_csv(cleaned_path / "football_matches_cleaned.csv", index=False)
country.to_csv(cleaned_path / "country_cleaned.csv", index=False)
league_clean.to_csv(cleaned_path / "league_cleaned.csv", index=False)
team.to_csv(cleaned_path / "team_cleaned.csv", index=False)

print("Cleaned files saved successfully.")

Cleaned files saved successfully.
